In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.linear_model import LogisticRegression
import numpy as np

# ======================================
# 1. LOAD DATASET
# ======================================

# File berlabel
df_labeled = pd.read_excel("data_label_manual.xlsx")

# File tidak berlabel
df_unlabeled = pd.read_excel("data750(cleaned).xlsx")

# Pastikan nama kolom sesuai file kamu
df_labeled.columns = ["id", "content", "sentiment"]
df_unlabeled.columns = ["id", "review_cleaned", "sentiment"]

# DROP kolom id
df_labeled = df_labeled.drop(columns=["id"])
df_unlabeled = df_unlabeled.drop(columns=["id"])

# Gabungkan dataset
df = pd.concat([
    df_labeled[["content", "sentiment"]].rename(columns={"content": "data"}),
    df_unlabeled[["review_cleaned", "sentiment"]].rename(columns={"review_cleaned": "data"})
], ignore_index=True)

# ======================================
# 2. PREPARE LABELS
# ======================================

df["sentiment"] = df["sentiment"].astype("object")

# Label kosong (unlabeled) diberi nilai -1
y = df["sentiment"].copy()
y_unlabeled = y.isna()
y_filled = y.copy()
y_filled[y_unlabeled] = -1   # unlabeled → -1

# ======================================
# 3. TF-IDF VECTORIZE
# ======================================

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df["data"])

# ======================================
# 4. SELF-TRAINING CLASSIFIER
# ======================================

base_model = LogisticRegression(max_iter=1000)

self_training = SelfTrainingClassifier(
    base_model,
    threshold=0.8,
    verbose=True
)

# Training semi-supervised
self_training.fit(X, y_filled)

# ======================================
# 5. PREDICT LABEL
# ======================================

pred_all = self_training.predict(X)
df["sentiment_self_training"] = pred_all

# ======================================
# 6. SAVE TO CSV
# ======================================

df.to_csv("hasil_self_training.csv", index=False)

print(df.tail(20))
print("Selesai! File disimpan sebagai hasil_self_training.csv")


End of iteration 1, added 142 new labels.
End of iteration 2, added 31 new labels.
End of iteration 3, added 12 new labels.
End of iteration 4, added 3 new labels.
End of iteration 5, added 15 new labels.
End of iteration 6, added 2 new labels.
                                                  data sentiment  \
980      game ini bagus tapi kenapa kayak yang ngetrol       NaN   
981                     game seru kalo main sama teman       NaN   
982                                               apik       NaN   
983                                    karna enakan ff       NaN   
984                                              bagus       NaN   
985                                      lumayan bagus       NaN   
986                                          bagus lhh       NaN   
987                                         game seruu       NaN   
988                                               puas       NaN   
989                 musuhnyq bot salim dulu sama kokoh       NaN   
990    